# Test statistics.py notebook:

This notebook is used to test the implementation of **jklab-core/statistics.py** module.

Testing (roughly) follows this routine:

0. import the module to test;
1. select component (variable/class/function) to test;
2. enstablish the correct behaviour;
3. implement a local function to assert the results;
4. summarise which tests have been passed.

## Module import

In [1]:
import numpy as np

import test_utils as jktu
import jklab.core.statistics as jkst

## Testing constants

In [2]:
# Collection of all test results in this notebook
test_results = {}

## Testing helpers

In [3]:
def reference_mean(
    data,
    axis=None,
    mask=None
):
    """
    Compute mean using an independent reference implementation.
    
    Parameters
    ----------
    data : array-like
        Input data.

    axis : int, tuple, or None, optional
        Axis or axes along which the mean is computed.

    mask : array-like of bool, optional
        Boolean mask selecting valid data.
    """
        
    # No axis
    if axis is None:

        total = 0
        count = 0

        for index in np.ndindex(data.shape):

            if mask is None or mask[index]:

                total += data[index]
                count += 1

        if count > 0:
            reference_result = total / count
        else:
            reference_result = np.nan
        
    # W/ axis
    else:

        if isinstance(axis, int):
            axes = (axis,)
        else:
            axes = axis

        remaining_axes = tuple(
            idx
            for idx in range(data.ndim)
            if idx not in axes
        )

        result_shape = tuple(
            data.shape[idx]
            for idx in remaining_axes
        )

        reference_result = np.empty(
            result_shape,
            dtype=float
        )

        for result_idx in np.ndindex(result_shape):

            total = 0
            count = 0

            for reduced_idx in np.ndindex(
                tuple(data.shape[idx] for idx in axes)
            ):

                data_idx = [None] * data.ndim

                for idx, value in zip(
                    remaining_axes,
                    result_idx
                ):
                    data_idx[idx] = value

                for idx, value in zip(
                    axes,
                    reduced_idx
                ):
                    data_idx[idx] = value

                data_idx = tuple(data_idx)

                if mask is None or mask[data_idx]:

                    total += data[data_idx]
                    count += 1

            if count > 0:
                reference_result[result_idx] = total / count
            else:
                reference_result[result_idx] = np.nan

    return reference_result

In [4]:
def reference_var(
    data,
    axis=None,
    mask=None,
    ddof=0
):
    """
    Compute variance using an independent reference implementation.

    Parameters
    ----------
    data : array-like
        Input data.

    axis : int, tuple, or None, optional
        Axis or axes along which the variance is computed.

    mask : array-like of bool, optional
        Boolean mask selecting valid data.

    ddof : int, optional
        Delta degrees of freedom.

    Returns
    -------
    float or numpy.ndarray
        Reference variance values.
    """

    data = np.asarray(data)

    # ==========================================================
    # Variance over all elements
    # ==========================================================

    if axis is None:

        total = 0
        count = 0

        for index in np.ndindex(data.shape):

            if mask is None or mask[index]:

                total += data[index]
                count += 1

        if count - ddof <= 0:
            return np.nan

        ref_mean = total / count

        squared_deviation = 0

        for index in np.ndindex(data.shape):

            if mask is None or mask[index]:

                squared_deviation += (
                    data[index] - ref_mean
                ) ** 2

        return squared_deviation / (count - ddof)

    # ==========================================================
    # Variance along one or more axes
    # ==========================================================

    if isinstance(axis, int):
        axes = (axis,)
    else:
        axes = axis

    remaining_axes = tuple(
        idx
        for idx in range(data.ndim)
        if idx not in axes
    )

    result_shape = tuple(
        data.shape[idx]
        for idx in remaining_axes
    )

    reference_result = np.empty(
        result_shape,
        dtype=float
    )

    reduced_shape = tuple(
        data.shape[idx]
        for idx in axes
    )

    for result_idx in np.ndindex(result_shape):

        total = 0
        count = 0

        # ------------------------------------------------------
        # Compute mean of the current reduced slice
        # ------------------------------------------------------

        for reduced_idx in np.ndindex(reduced_shape):

            data_idx = [None] * data.ndim

            for idx, value in zip(
                remaining_axes,
                result_idx
            ):
                data_idx[idx] = value

            for idx, value in zip(
                axes,
                reduced_idx
            ):
                data_idx[idx] = value

            data_idx = tuple(data_idx)

            if mask is None or mask[data_idx]:

                total += data[data_idx]
                count += 1

        if count - ddof <= 0:

            reference_result[result_idx] = np.nan

            continue

        ref_mean = total / count

        # ------------------------------------------------------
        # Compute squared deviations
        # ------------------------------------------------------

        squared_deviation = 0

        for reduced_idx in np.ndindex(reduced_shape):

            data_idx = [None] * data.ndim

            for idx, value in zip(
                remaining_axes,
                result_idx
            ):
                data_idx[idx] = value

            for idx, value in zip(
                axes,
                reduced_idx
            ):
                data_idx[idx] = value

            data_idx = tuple(data_idx)

            if mask is None or mask[data_idx]:

                squared_deviation += (
                    data[data_idx] - ref_mean
                ) ** 2

        reference_result[result_idx] = (
            squared_deviation / (count - ddof)
        )

    return reference_result

In [5]:
def reference_std(
    data,
    axis=None,
    mask=None,
    ddof=0
):
    reference_result = reference_var(
        data=data,
        axis=axis,
        mask=mask,
        ddof=ddof
    )

    return np.sqrt(reference_result)

In [6]:
def reference_sem(
    data,
    axis=None,
    mask=None
):
    """
    Compute standard error of the mean using an independent
    reference implementation.

    Parameters
    ----------
    data : array-like
        Input data.

    axis : int, tuple, or None, optional
        Axis or axes along which the SEM is computed.

    mask : array-like of bool, optional
        Boolean mask selecting valid data.

    Returns
    -------
    float or numpy.ndarray
        Reference SEM values.
    """

    data = np.asarray(data)

    # ==========================================================
    # SEM over all elements
    # ==========================================================

    if axis is None:

        total = 0
        count = 0

        for index in np.ndindex(data.shape):

            if mask is None or mask[index]:

                if not np.isnan(data[index]):

                    total += data[index]
                    count += 1

        if count <= 1:
            return np.nan

        ref_mean = total / count

        squared_deviation = 0

        for index in np.ndindex(data.shape):

            if mask is None or mask[index]:

                if not np.isnan(data[index]):

                    squared_deviation += (
                        data[index] - ref_mean
                    ) ** 2

        sample_var = (
            squared_deviation / (count - 1)
        )

        sample_std = np.sqrt(
            sample_var
        )

        return sample_std / np.sqrt(count)

    # ==========================================================
    # SEM along one or more axes
    # ==========================================================

    if isinstance(axis, int):
        axes = (axis,)
    else:
        axes = axis

    remaining_axes = tuple(
        idx
        for idx in range(data.ndim)
        if idx not in axes
    )

    result_shape = tuple(
        data.shape[idx]
        for idx in remaining_axes
    )

    reference_result = np.empty(
        result_shape,
        dtype=float
    )

    reduced_shape = tuple(
        data.shape[idx]
        for idx in axes
    )

    for result_idx in np.ndindex(result_shape):

        total = 0
        count = 0

        # ------------------------------------------------------
        # Compute mean of current reduced slice
        # ------------------------------------------------------

        for reduced_idx in np.ndindex(reduced_shape):

            data_idx = [None] * data.ndim

            for idx, value in zip(
                remaining_axes,
                result_idx
            ):
                data_idx[idx] = value

            for idx, value in zip(
                axes,
                reduced_idx
            ):
                data_idx[idx] = value

            data_idx = tuple(data_idx)

            if mask is None or mask[data_idx]:

                if not np.isnan(data[data_idx]):

                    total += data[data_idx]
                    count += 1

        if count <= 1:

            reference_result[result_idx] = np.nan

            continue

        ref_mean = total / count

        # ------------------------------------------------------
        # Compute squared deviations
        # ------------------------------------------------------

        squared_deviation = 0

        for reduced_idx in np.ndindex(reduced_shape):

            data_idx = [None] * data.ndim

            for idx, value in zip(
                remaining_axes,
                result_idx
            ):
                data_idx[idx] = value

            for idx, value in zip(
                axes,
                reduced_idx
            ):
                data_idx[idx] = value

            data_idx = tuple(data_idx)

            if mask is None or mask[data_idx]:

                if not np.isnan(data[data_idx]):

                    squared_deviation += (
                        data[data_idx] - ref_mean
                    ) ** 2

        sample_var = (
            squared_deviation / (count - 1)
        )

        sample_std = np.sqrt(
            sample_var
        )

        reference_result[result_idx] = (
            sample_std / np.sqrt(count)
        )

    return reference_result

# Test 1 - _apply_mask()

## - Automated testing

In [7]:
def test_apply_mask(
    data,
    mask=None
):
    """
    Test the _apply_mask function.
    """

    # Build expected result manually.
    if mask is None:

        expected = np.asarray(data)

    else:

        expected = np.empty_like(
            data,
            dtype=float
        )

        for index in np.ndindex(data.shape):

            if mask[index]:
                expected[index] = data[index]

            else:
                expected[index] = np.nan

    # Apply mask via JKateLab.
    result = jkst._apply_mask(
        data=data,
        mask=mask
    )

    # Compare results.
    test = np.array_equal(
        expected,
        result,
        equal_nan=True
    )

    return test

### 1. Correct masking

In [8]:
# === Input ===
# Pick an array-like and a mask.
test_data = np.array([-1., 0., 1.], dtype=float)
test_mask = test_data > 0

# === Test & Record ===
jktu.record_test(
    test_results=test_results,
    test_name="_apply_mask: correct masking",
    condition=test_apply_mask(
        data=test_data,
        mask=test_mask
    )
)

✅ PASSED: _apply_mask: correct masking.


### 2. No masking

In [9]:
# === Input ===
# Pick an array-like and no mask.
test_data = np.array([-1., 0., 1.], dtype=float)
test_mask = None

# === Test & Record ===
jktu.record_test(
    test_results=test_results,
    test_name="_apply_mask: no masking",
    condition=test_apply_mask(
        data=test_data,
        mask=test_mask
    )
)

✅ PASSED: _apply_mask: no masking.


### 3. Multidimensional masking

In [10]:
# === Input ===
# Pick an array-like and no mask.
test_data = np.array([
    [-1., 0., 1.],
    [1., 2., 3.],
    [-1., -2., -3.],
], dtype=float)
test_mask = test_data > 0

# === Test & Record ===
jktu.record_test(
    test_results=test_results,
    test_name="_apply_mask: multidimensional masking",
    condition=test_apply_mask(
        data=test_data,
        mask=test_mask
    )
)

✅ PASSED: _apply_mask: multidimensional masking.


### 4. Integer data masking

In [11]:
# === Input ===
# Pick an array-like and a mask.
test_data = np.array([-1, 0, 1], dtype=int)
test_mask = test_data > 0

# === Test & Record ===
jktu.record_test(
    test_results=test_results,
    test_name="_apply_mask: correct masking",
    condition=test_apply_mask(
        data=test_data,
        mask=test_mask
    )
)

✅ PASSED: _apply_mask: correct masking.


## - Visual testing

In [12]:
test_array = np.array([
    [0., -2., 3.],
    [-1., 2., -3.]
], dtype=float)

expected_array = np.array([
    [np.nan, np.nan, 3.],
    [np.nan, 2., np.nan]
], dtype=float)

result_array = jkst._apply_mask(
        data=expected_array,
        mask=expected_array > 0
)

print(result_array)
print(np.array_equal(
    result_array,
    expected_array,
    equal_nan=True
))

[[nan nan  3.]
 [nan  2. nan]]
True


In [13]:
test_array = np.array([
    [0., -2., 3.],
    [-1., 2., -3.]
], dtype=float)

test_mask = None

expected_array = np.where(
    test_mask,
    test_array,
    np.nan
)

result_array = jkst._apply_mask(
        data=expected_array,
)

print(result_array)
print("-")
print(expected_array)
print(np.array_equal(
    result_array,
    expected_array,
    equal_nan=True
))

[[nan nan nan]
 [nan nan nan]]
-
[[nan nan nan]
 [nan nan nan]]
True


In [14]:
test_array_int = np.array([
    [0, 2, 3],
    [-1, 2, -3]
], dtype=int)

result_array = jkst._apply_mask(
        data=test_array_int,
        mask=None
)

print(result_array)
print(result_array.dtype)

[[ 0  2  3]
 [-1  2 -3]]
int64


# Test 2 - compute_mean()

## - Automated testing

In [15]:
def test_compute_mean(
    data,
    axis=None,
    mask=None
):
    """
    Test compute_mean() function.
    """

    # Reference computation.
    reference_result = reference_mean(
        data=data,
        axis=axis,
        mask=mask
    )

    # Average via jklab
    jklab_result = jkst.compute_mean(
        data=data,
        axis=axis,
        mask=mask
    )

    # Compare
    test = np.allclose(
        reference_result,
        jklab_result
    )

    return test

### 1. 1D array, no axis, no mask

In [16]:
# === Input ===
# Pick a 1D array-like
test_data = np.array([-1., 0., 1., 1.])

# === Test & Record ===
jktu.record_test(
    test_results=test_results,
    test_name="compute_mean: 1d array, no axis, no mask",
    condition=test_compute_mean(
        data=test_data
    )
)

✅ PASSED: compute_mean: 1d array, no axis, no mask.


### 2. 1D array, no axis, w/ mask

In [17]:
# === Input ===
# Pick a 1D array-like and a mask
test_data = np.array([-1., 0., 1., 1.])
test_mask = test_data > 0

# === Test & Record ===
jktu.record_test(
    test_results=test_results,
    test_name="compute_mean: 1d array, no axis, w/ mask",
    condition=test_compute_mean(
        data=test_data,
        mask=test_mask
    )
)

✅ PASSED: compute_mean: 1d array, no axis, w/ mask.


### 3. ND array, no axis, w/ mask

In [18]:
# === Input ===
# Pick an ND array-like and a mask
test_data = np.array([
    [-1., 0., 1.],
    [1., 2., 3.],
    [-1., -2., -3.],
], dtype=float)
test_mask = test_data > 0

# === Test & Record ===
jktu.record_test(
    test_results=test_results,
    test_name="compute_mean: nd array, no axis, w/ mask",
    condition=test_compute_mean(
        data=test_data,
        mask=test_mask
    )
)

✅ PASSED: compute_mean: nd array, no axis, w/ mask.


### 4. ND array, w/ axis, no mask

In [19]:
# === Input ===
# Pick an ND array-like and an axis
test_data = np.array([
    [-1., 0., 1.],
    [1., 2., 3.],
    [-1., -2., -3.],
], dtype=float)
test_axis = 0

# === Test & Record ===
jktu.record_test(
    test_results=test_results,
    test_name="compute_mean: nd array, w/ axis, no mask",
    condition=test_compute_mean(
        data=test_data,
        axis=test_axis
    )
)

✅ PASSED: compute_mean: nd array, w/ axis, no mask.


### 5. ND array, w/ axis, w/ mask

In [20]:
# === Input ===
# Pick an ND array-like, an axis and a mask
test_data = np.array([
    [-1., 0., 1.],
    [1., 2., 3.],
    [-1., -2., -3.],
], dtype=float)
test_axis = 0
test_mask = test_data > 0

# === Test & Record ===
jktu.record_test(
    test_results=test_results,
    test_name="compute_mean: nd array, w/ axis, w/ mask",
    condition=test_compute_mean(
        data=test_data,
        axis=test_axis,
        mask=test_mask
    )
)

✅ PASSED: compute_mean: nd array, w/ axis, w/ mask.


### 6. ND array, w/ axis tuple, no mask

In [21]:
# === Input ===
# Pick an ND array-like and a tuple of axis
test_data = np.array([
    [
        [-1., 0., 1.],
        [1., 2., 3.],
        [-1., -2., -3.],
    ],
    [
        [-1., 0., 1.],
        [1., 2., 3.],
        [-1., -2., -3.],
    ]
], dtype=float)
test_axis = (0,1)

# === Test & Record ===
jktu.record_test(
    test_results=test_results,
    test_name="compute_mean: nd array, w/ tuple axis, no mask",
    condition=test_compute_mean(
        data=test_data,
        axis=test_axis
    )
)

✅ PASSED: compute_mean: nd array, w/ tuple axis, no mask.


### 7. ND array, w/ axis tuple, w/ mask

In [22]:
# === Input ===
# Pick an ND array-like and a tuple of axis
test_data = np.array([
    [
        [-1., 0., 1.],
        [1., 2., 3.],
        [-1., -2., -3.],
    ],
    [
        [-1., 0., 1.],
        [1., 2., 3.],
        [-1., -2., -3.],
    ]
], dtype=float)
test_axis = (0,1)
test_mask = test_data > 0

# === Test & Record ===
jktu.record_test(
    test_results=test_results,
    test_name="compute_mean: nd array, w/ tuple axis, w/ mask",
    condition=test_compute_mean(
        data=test_data,
        axis=test_axis,
        mask=test_mask
    )
)

✅ PASSED: compute_mean: nd array, w/ tuple axis, w/ mask.


## - Visual testing

In [23]:
# === Input ===
# Pick an ND array-like and a tuple of axis
test_data = np.array([
    [
        [-1., 0., 1.],
        [1., 2., 3.],
        [-1., -2., -3.],
    ],
    [
        [-1., 0., 1.],
        [1., 2., 3.],
        [-1., -2., -3.],
    ]
], dtype=float)
test_axis = (0,1)

print(test_data.shape)

jklab_result = jkst.compute_mean(
    data=test_data,
    axis=test_axis
)

print(jklab_result.shape)

print(jklab_result)
print(np.mean(test_data, axis=test_axis))

(2, 3, 3)
(3,)
[-0.33333333  0.          0.33333333]
[-0.33333333  0.          0.33333333]


In [24]:
# ND array w/ axis, w/ mask
test_data = np.array([
    [-1., 0., 1.],
    [1., 2., 3.],
    [-1., -2., -3.],
], dtype=float)
test_axis = 0
test_mask = test_data > 0


numpy_result = np.mean(
    test_data[test_mask],
    axis=test_axis
)

jklab_result = jkst.compute_mean(
    data=test_data,
    axis=test_axis,
    mask=test_mask
)

print(f"numpy: {numpy_result}")
print(f"jklab: {jklab_result}")

numpy: 1.75
jklab: [1. 2. 2.]


In [25]:
# 1D array
test_array = np.array([0., 0., 0., 1.])
test_mask = test_array > 0

if test_mask is None:
    masked_array = test_array
else:
    masked_array = np.where(test_mask, test_array, np.nan)


result_nonan = np.mean(test_array)
expected_result = np.nanmean(masked_array)
masked_result = jkst.compute_mean(
    data=test_array,
    mask=test_mask
)

print(result_nonan)
print(expected_result, masked_result)

0.25
1.0 1.0


In [26]:
test_data = np.array([
    [-1., 0., 1.],
    [1., 2., 3.],
    [-1., -2., -3.],
    [5., 2., -4.]
], dtype=float)
test_axis = 0

print(test_data.shape)

print(np.mean(test_data, axis=0), jkst.compute_mean(test_data, axis=0))

(4, 3)
[ 1.    0.5  -0.75] [ 1.    0.5  -0.75]


In [27]:
# Pick an ND array-like and a mask
test_data = np.array([
    [-1., 0., 1.],
    [1., 2., 3.],
    [-1., -2., -3.],
], dtype=float)
test_mask = test_data > 0

numpy_result = np.mean(test_data[test_mask])

jklab_result = jkst.compute_mean(
    data=test_data,
    mask=test_mask
)

print(numpy_result, jklab_result)

1.75 1.75


# Test 3 - compute_var()

## - Automated testing

In [28]:
def test_compute_var(
    data,
    axis=None,
    mask=None,
    ddof=0,
    verbose=False,
):
    """
    Test compute_var() function.
    """

    # Reference computation.
    reference_result = reference_var(
        data=data,
        axis=axis,
        mask=mask,
        ddof=ddof
    )

    # Average via jklab
    jklab_result = jkst.compute_var(
        data=data,
        axis=axis,
        mask=mask,
        ddof=ddof
    )

    # Compare
    test = np.allclose(
        reference_result,
        jklab_result,
        equal_nan=True
    )

    if verbose:
        print(f"reference : {reference_result}")
        print(f"jklab     : {jklab_result}")

    return test

### 1. 1D array, no axis, no mask

In [29]:
# === Input ===
# Pick a 1D array-like
test_data = np.array([-1., 0., 1., 1.])

# === Test & Record ===
jktu.record_test(
    test_results=test_results,
    test_name="compute_var: 1d array, no axis, no mask",
    condition=test_compute_var(
        data=test_data
    )
)

✅ PASSED: compute_var: 1d array, no axis, no mask.


### 2. 1D array, no axis, w/ mask

In [30]:
# === Input ===
# Pick a 1D array-like and a mask
test_data = np.array([-1., 0., 1., 1.])
test_mask = test_data > 0

# === Test & Record ===
jktu.record_test(
    test_results=test_results,
    test_name="compute_var: 1d array, no axis, w/ mask",
    condition=test_compute_var(
        data=test_data,
        mask=test_mask
    )
)

✅ PASSED: compute_var: 1d array, no axis, w/ mask.


### 3. ND array, no axis, w/ mask

In [31]:
# === Input ===
# Pick an ND array-like and a mask
test_data = np.array([
    [-1., 0., 1.],
    [1., 2., 3.],
    [-1., -2., -3.],
], dtype=float)
test_mask = test_data > 0

# === Test & Record ===
jktu.record_test(
    test_results=test_results,
    test_name="compute_var: nd array, no axis, w/ mask",
    condition=test_compute_var(
        data=test_data,
        mask=test_mask
    )
)

✅ PASSED: compute_var: nd array, no axis, w/ mask.


### 4. ND array, w/ axis, no mask

In [32]:
# === Input ===
# Pick an ND array-like and an axis
test_data = np.array([
    [-1., 0., 1.],
    [1., 2., 3.],
    [-1., -2., -3.],
], dtype=float)
test_axis = 0

# === Test & Record ===
jktu.record_test(
    test_results=test_results,
    test_name="compute_var: nd array, w/ axis, no mask",
    condition=test_compute_var(
        data=test_data,
        axis=test_axis
    )
)

✅ PASSED: compute_var: nd array, w/ axis, no mask.


### 5. ND array, w/ axis, w/ mask

In [33]:
# === Input ===
# Pick an ND array-like, an axis and a mask
test_data = np.array([
    [-1., 0., 1.],
    [1., 2., 3.],
    [-1., -2., -3.],
], dtype=float)
test_axis = 0
test_mask = test_data > 0

# === Test & Record ===
jktu.record_test(
    test_results=test_results,
    test_name="compute_var: nd array, w/ axis, w/ mask",
    condition=test_compute_var(
        data=test_data,
        axis=test_axis,
        mask=test_mask
    )
)

✅ PASSED: compute_var: nd array, w/ axis, w/ mask.


### 6. ND array, w/ axis tuple, no mask

In [34]:
# === Input ===
# Pick an ND array-like and a tuple of axis
test_data = np.array([
    [
        [-1., 0., 1.],
        [1., 2., 3.],
        [-1., -2., -3.],
    ],
    [
        [-1., 0., 1.],
        [1., 2., 3.],
        [-1., -2., -3.],
    ]
], dtype=float)
test_axis = (0,1)

# === Test & Record ===
jktu.record_test(
    test_results=test_results,
    test_name="compute_var: nd array, w/ tuple axis, no mask",
    condition=test_compute_var(
        data=test_data,
        axis=test_axis
    )
)

✅ PASSED: compute_var: nd array, w/ tuple axis, no mask.


### 7. ND array, w/ axis tuple, w/ mask

In [35]:
# === Input ===
# Pick an ND array-like and a tuple of axis
test_data = np.array([
    [
        [-1., 0., 1.],
        [1., 2., 3.],
        [-1., -2., -3.],
    ],
    [
        [-1., 0., 1.],
        [1., 2., 3.],
        [-1., -2., -3.],
    ]
], dtype=float)
test_axis = (0,1)
test_mask = test_data > 0

# === Test & Record ===
jktu.record_test(
    test_results=test_results,
    test_name="compute_var: nd array, w/ tuple axis, w/ mask",
    condition=test_compute_var(
        data=test_data,
        axis=test_axis,
        mask=test_mask
    )
)

✅ PASSED: compute_var: nd array, w/ tuple axis, w/ mask.


### 8. ND array, no axis, no mask, ddof=1

In [36]:
# === Input ===
# Pick an ND array-like and a tuple of axis
test_data = np.array([
    [
        [-1., 0., 1.],
        [1., 2., 3.],
        [-1., -2., -3.],
    ],
    [
        [-1., 0., 1.],
        [1., 2., 3.],
        [-1., -2., -3.],
    ]
], dtype=float)
test_ddof = 1

# === Test & Record ===
jktu.record_test(
    test_results=test_results,
    test_name="compute_var: nd array, no axis, no mask, ddof = 1",
    condition=test_compute_var(
        data=test_data,
        ddof=test_ddof
    )
)

✅ PASSED: compute_var: nd array, no axis, no mask, ddof = 1.


### 9. ND array, no axis tuple, no mask, ddof=n_valid-1

In [37]:
# === Input ===
# Pick an ND array-like and a tuple of axis
test_data = np.array([
    [
        [-1., 0., 1.],
        [1., 2., 3.],
        [-1., -2., -3.],
    ],
    [
        [-1., 0., 1.],
        [1., 2., 3.],
        [-1., -2., -3.],
    ]
], dtype=float)
test_ddof = test_data.size - 1

# === Test & Record ===
jktu.record_test(
    test_results=test_results,
    test_name=("compute_var: nd array, no axis, "
               f"no mask, ddof=n_valid-1"),
    condition=test_compute_var(
        data=test_data,
        ddof=test_ddof
    )
)

✅ PASSED: compute_var: nd array, no axis, no mask, ddof=n_valid-1.


### 10. ND array, no axis tuple, no mask, ddof=n_valid

In [38]:
# === Input ===
# Pick an ND array-like and a tuple of axis
test_data = np.array([
    [
        [-1., 0., 1.],
        [1., 2., 3.],
        [-1., -2., -3.],
    ],
    [
        [-1., 0., 1.],
        [1., 2., 3.],
        [-1., -2., -3.],
    ]
], dtype=float)
test_ddof = test_data.size

# === Test & Record ===
jktu.record_test(
    test_results=test_results,
    test_name=("compute_var: nd array, no axis, "
               f"no mask, ddof=n_valid"),
    condition=test_compute_var(
        data=test_data,
        ddof=test_ddof
    )
)

✅ PASSED: compute_var: nd array, no axis, no mask, ddof=n_valid.


/Users/menny/Code/myPython/JKateLab/jklab-core/src/jklab/core/statistics.py:159: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var_value = np.nanvar(


### 11. ND array, no axis tuple, no mask, ddof=n_valid+1

In [39]:
# === Input ===
# Pick an ND array-like and a tuple of axis
test_data = np.array([
    [
        [-1., 0., 1.],
        [1., 2., 3.],
        [-1., -2., -3.],
    ],
    [
        [-1., 0., 1.],
        [1., 2., 3.],
        [-1., -2., -3.],
    ]
], dtype=float)
test_ddof = test_data.size + 1

# === Test & Record ===
jktu.record_test(
    test_results=test_results,
    test_name=("compute_var: nd array, no axis, "
               f"no mask, ddof=n_valid+1"),
    condition=test_compute_var(
        data=test_data,
        ddof=test_ddof
    )
)

✅ PASSED: compute_var: nd array, no axis, no mask, ddof=n_valid+1.


# Test 4 - compute_std()

## - Automated testing

In [40]:
def test_compute_std(
    data,
    axis=None,
    mask=None,
    ddof=0,
    verbose=False,
):
    """
    Test compute_std() function.
    """

    # Reference computation.
    reference_result = reference_std(
        data=data,
        axis=axis,
        mask=mask,
        ddof=ddof
    )

    # Average via jklab
    jklab_result = jkst.compute_std(
        data=data,
        axis=axis,
        mask=mask,
        ddof=ddof
    )

    # Compare
    test = np.allclose(
        reference_result,
        jklab_result,
        equal_nan=True
    )

    if verbose:
        print(f"reference : {reference_result}")
        print(f"jklab     : {jklab_result}")

    return test

### 1. 1D array, no axis, no mask

In [41]:
# === Input ===
# Pick a 1D array-like
test_data = np.array([-1., 0., 1., 1.])

# === Test & Record ===
jktu.record_test(
    test_results=test_results,
    test_name="compute_std: 1d array, no axis, no mask",
    condition=test_compute_std(
        data=test_data
    )
)

✅ PASSED: compute_std: 1d array, no axis, no mask.


### 2. 1D array, no axis, w/ mask

In [42]:
# === Input ===
# Pick a 1D array-like and a mask
test_data = np.array([-1., 0., 1., 1.])
test_mask = test_data > 0

# === Test & Record ===
jktu.record_test(
    test_results=test_results,
    test_name="compute_std: 1d array, no axis, w/ mask",
    condition=test_compute_std(
        data=test_data,
        mask=test_mask
    )
)

✅ PASSED: compute_std: 1d array, no axis, w/ mask.


### 3. ND array, no axis, w/ mask

In [43]:
# === Input ===
# Pick an ND array-like and a mask
test_data = np.array([
    [-1., 0., 1.],
    [1., 2., 3.],
    [-1., -2., -3.],
], dtype=float)
test_mask = test_data > 0

# === Test & Record ===
jktu.record_test(
    test_results=test_results,
    test_name="compute_std: nd array, no axis, w/ mask",
    condition=test_compute_std(
        data=test_data,
        mask=test_mask
    )
)

✅ PASSED: compute_std: nd array, no axis, w/ mask.


### 4. ND array, w/ axis, no mask

In [44]:
# === Input ===
# Pick an ND array-like and an axis
test_data = np.array([
    [-1., 0., 1.],
    [1., 2., 3.],
    [-1., -2., -3.],
], dtype=float)
test_axis = 0

# === Test & Record ===
jktu.record_test(
    test_results=test_results,
    test_name="compute_std: nd array, w/ axis, no mask",
    condition=test_compute_std(
        data=test_data,
        axis=test_axis
    )
)

✅ PASSED: compute_std: nd array, w/ axis, no mask.


### 5. ND array, w/ axis, w/ mask

In [45]:
# === Input ===
# Pick an ND array-like, an axis and a mask
test_data = np.array([
    [-1., 0., 1.],
    [1., 2., 3.],
    [-1., -2., -3.],
], dtype=float)
test_axis = 0
test_mask = test_data > 0

# === Test & Record ===
jktu.record_test(
    test_results=test_results,
    test_name="compute_std: nd array, w/ axis, w/ mask",
    condition=test_compute_std(
        data=test_data,
        axis=test_axis,
        mask=test_mask
    )
)

✅ PASSED: compute_std: nd array, w/ axis, w/ mask.


### 6. ND array, w/ axis tuple, no mask

In [46]:
# === Input ===
# Pick an ND array-like and a tuple of axis
test_data = np.array([
    [
        [-1., 0., 1.],
        [1., 2., 3.],
        [-1., -2., -3.],
    ],
    [
        [-1., 0., 1.],
        [1., 2., 3.],
        [-1., -2., -3.],
    ]
], dtype=float)
test_axis = (0,1)

# === Test & Record ===
jktu.record_test(
    test_results=test_results,
    test_name="compute_std: nd array, w/ tuple axis, no mask",
    condition=test_compute_std(
        data=test_data,
        axis=test_axis
    )
)

✅ PASSED: compute_std: nd array, w/ tuple axis, no mask.


### 7. ND array, w/ axis tuple, w/ mask

In [47]:
# === Input ===
# Pick an ND array-like and a tuple of axis
test_data = np.array([
    [
        [-1., 0., 1.],
        [1., 2., 3.],
        [-1., -2., -3.],
    ],
    [
        [-1., 0., 1.],
        [1., 2., 3.],
        [-1., -2., -3.],
    ]
], dtype=float)
test_axis = (0,1)
test_mask = test_data > 0

# === Test & Record ===
jktu.record_test(
    test_results=test_results,
    test_name="compute_std: nd array, w/ tuple axis, w/ mask",
    condition=test_compute_std(
        data=test_data,
        axis=test_axis,
        mask=test_mask
    )
)

✅ PASSED: compute_std: nd array, w/ tuple axis, w/ mask.


### 8. ND array, no axis, no mask, ddof=1

In [48]:
# === Input ===
# Pick an ND array-like and a tuple of axis
test_data = np.array([
    [
        [-1., 0., 1.],
        [1., 2., 3.],
        [-1., -2., -3.],
    ],
    [
        [-1., 0., 1.],
        [1., 2., 3.],
        [-1., -2., -3.],
    ]
], dtype=float)
test_ddof = 1

# === Test & Record ===
jktu.record_test(
    test_results=test_results,
    test_name="compute_std: nd array, no axis, no mask, ddof = 1",
    condition=test_compute_std(
        data=test_data,
        ddof=test_ddof
    )
)

✅ PASSED: compute_std: nd array, no axis, no mask, ddof = 1.


### 9. ND array, no axis tuple, no mask, ddof=n_valid-1

In [49]:
# === Input ===
# Pick an ND array-like and a tuple of axis
test_data = np.array([
    [
        [-1., 0., 1.],
        [1., 2., 3.],
        [-1., -2., -3.],
    ],
    [
        [-1., 0., 1.],
        [1., 2., 3.],
        [-1., -2., -3.],
    ]
], dtype=float)
test_ddof = test_data.size - 1

# === Test & Record ===
jktu.record_test(
    test_results=test_results,
    test_name=("compute_std: nd array, no axis, "
               f"no mask, ddof=n_valid-1"),
    condition=test_compute_std(
        data=test_data,
        ddof=test_ddof
    )
)

✅ PASSED: compute_std: nd array, no axis, no mask, ddof=n_valid-1.


### 10. ND array, no axis tuple, no mask, ddof=n_valid

In [50]:
# === Input ===
# Pick an ND array-like and a tuple of axis
test_data = np.array([
    [
        [-1., 0., 1.],
        [1., 2., 3.],
        [-1., -2., -3.],
    ],
    [
        [-1., 0., 1.],
        [1., 2., 3.],
        [-1., -2., -3.],
    ]
], dtype=float)
test_ddof = test_data.size

# === Test & Record ===
jktu.record_test(
    test_results=test_results,
    test_name=("compute_std: nd array, no axis, "
               f"no mask, ddof=n_valid"),
    condition=test_compute_std(
        data=test_data,
        ddof=test_ddof
    )
)

✅ PASSED: compute_std: nd array, no axis, no mask, ddof=n_valid.


/Users/menny/Code/myPython/JKateLab/jklab-core/.venv/lib/python3.14/site-packages/numpy/lib/_nanfunctions_impl.py:1997: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,


### 11. ND array, no axis tuple, no mask, ddof=n_valid+1

In [51]:
# === Input ===
# Pick an ND array-like and a tuple of axis
test_data = np.array([
    [
        [-1., 0., 1.],
        [1., 2., 3.],
        [-1., -2., -3.],
    ],
    [
        [-1., 0., 1.],
        [1., 2., 3.],
        [-1., -2., -3.],
    ]
], dtype=float)
test_ddof = test_data.size + 1

# === Test & Record ===
jktu.record_test(
    test_results=test_results,
    test_name=("compute_std: nd array, no axis, "
               f"no mask, ddof=n_valid+1"),
    condition=test_compute_std(
        data=test_data,
        ddof=test_ddof
    )
)

✅ PASSED: compute_std: nd array, no axis, no mask, ddof=n_valid+1.


# Test 5 - compute_sem()

## - Automated testing

In [52]:
def test_compute_sem(
    data,
    axis=None,
    mask=None,
    verbose=False,
):
    """
    Test compute_sem() function.
    """

    # Reference computation.
    reference_result = reference_sem(
        data=data,
        axis=axis,
        mask=mask,
    )

    # Average via jklab
    jklab_result = jkst.compute_sem(
        data=data,
        axis=axis,
        mask=mask,
    )

    # Compare
    test = np.allclose(
        reference_result,
        jklab_result,
        equal_nan=True
    )

    if verbose:
        print(f"reference : {reference_result}")
        print(f"jklab     : {jklab_result}")

    return test

### 1. 1D array, no axis, no mask

In [53]:
# === Input ===
# Pick a 1D array-like
test_data = np.array([-1., 0., 1., 1.])

# === Test & Record ===
jktu.record_test(
    test_results=test_results,
    test_name="compute_sem: 1d array, no axis, no mask",
    condition=test_compute_sem(
        data=test_data
    )
)

✅ PASSED: compute_sem: 1d array, no axis, no mask.


### 2. 1D array, no axis, w/ mask

In [54]:
# === Input ===
# Pick a 1D array-like and a mask
test_data = np.array([-1., 0., 1., 1.])
test_mask = test_data > 0

# === Test & Record ===
jktu.record_test(
    test_results=test_results,
    test_name="compute_sem: 1d array, no axis, w/ mask",
    condition=test_compute_sem(
        data=test_data,
        mask=test_mask
    )
)

✅ PASSED: compute_sem: 1d array, no axis, w/ mask.


### 3. ND array, no axis, w/ mask

In [55]:
# === Input ===
# Pick an ND array-like and a mask
test_data = np.array([
    [-1., 0., 1.],
    [1., 2., 3.],
    [-1., -2., -3.],
], dtype=float)
test_mask = test_data > 0

# === Test & Record ===
jktu.record_test(
    test_results=test_results,
    test_name="compute_sem: nd array, no axis, w/ mask",
    condition=test_compute_sem(
        data=test_data,
        mask=test_mask
    )
)

✅ PASSED: compute_sem: nd array, no axis, w/ mask.


### 4. ND array, w/ axis, no mask

In [56]:
# === Input ===
# Pick an ND array-like and an axis
test_data = np.array([
    [-1., 0., 1.],
    [1., 2., 3.],
    [-1., -2., -3.],
], dtype=float)
test_axis = 0

# === Test & Record ===
jktu.record_test(
    test_results=test_results,
    test_name="compute_sem: nd array, w/ axis, no mask",
    condition=test_compute_sem(
        data=test_data,
        axis=test_axis
    )
)

✅ PASSED: compute_sem: nd array, w/ axis, no mask.


### 5. ND array, w/ axis, w/ mask

In [57]:
# === Input ===
# Pick an ND array-like, an axis and a mask
test_data = np.array([
    [-1., 0., 1.],
    [1., 2., 3.],
    [-1., -2., -3.],
], dtype=float)
test_axis = 0
test_mask = test_data > 0

# === Test & Record ===
jktu.record_test(
    test_results=test_results,
    test_name="compute_sem: nd array, w/ axis, w/ mask",
    condition=test_compute_sem(
        data=test_data,
        axis=test_axis,
        mask=test_mask
    )
)

✅ PASSED: compute_sem: nd array, w/ axis, w/ mask.


### 6. ND array, w/ axis tuple, no mask

In [58]:
# === Input ===
# Pick an ND array-like and a tuple of axis
test_data = np.array([
    [
        [-1., 0., 1.],
        [1., 2., 3.],
        [-1., -2., -3.],
    ],
    [
        [-1., 0., 1.],
        [1., 2., 3.],
        [-1., -2., -3.],
    ]
], dtype=float)
test_axis = (0,1)

# === Test & Record ===
jktu.record_test(
    test_results=test_results,
    test_name="compute_sem: nd array, w/ tuple axis, no mask",
    condition=test_compute_sem(
        data=test_data,
        axis=test_axis
    )
)

✅ PASSED: compute_sem: nd array, w/ tuple axis, no mask.


### 7. ND array, w/ axis tuple, w/ mask

In [61]:
# === Input ===
# Pick an ND array-like and a tuple of axis
test_data = np.array([
    [
        [-1., 0., 1.],
        [1., 2., 3.],
        [-1., -2., -3.],
    ],
    [
        [-1., 0., 1.],
        [1., 2., 3.],
        [-1., -2., -3.],
    ]
], dtype=float)
test_axis = (0,1)
test_mask = test_data > 0

# === Test & Record ===
jktu.record_test(
    test_results=test_results,
    test_name="compute_sem: nd array, w/ tuple axis, w/ mask",
    condition=test_compute_sem(
        data=test_data,
        axis=test_axis,
        mask=test_mask
    )
)

✅ PASSED: compute_sem: nd array, w/ tuple axis, w/ mask.


# Summary

In [62]:
jktu.print_test_summary(test_results)


TEST SUMMARY
Passed: 39/39
Failed: 0/39
Success rate: 100.0%

PASSED: _apply_mask: correct masking
PASSED: _apply_mask: no masking
PASSED: _apply_mask: multidimensional masking
PASSED: compute_mean: 1d array, no axis, no mask
PASSED: compute_mean: 1d array, no axis, w/ mask
PASSED: compute_mean: nd array, no axis, w/ mask
PASSED: compute_mean: nd array, w/ axis, no mask
PASSED: compute_mean: nd array, w/ axis, w/ mask
PASSED: compute_mean: nd array, w/ tuple axis, no mask
PASSED: compute_mean: nd array, w/ tuple axis, w/ mask
PASSED: compute_var: 1d array, no axis, no mask
PASSED: compute_var: 1d array, no axis, w/ mask
PASSED: compute_var: nd array, no axis, w/ mask
PASSED: compute_var: nd array, w/ axis, no mask
PASSED: compute_var: nd array, w/ axis, w/ mask
PASSED: compute_var: nd array, w/ tuple axis, no mask
PASSED: compute_var: nd array, w/ tuple axis, w/ mask
PASSED: compute_var: nd array, no axis, no mask, ddof = 1
PASSED: compute_var: nd array, no axis, no mask, ddof=n_valid